# 05. 입력 길이에 따라 정답을 활용하는 성능이 달라질까요?

같은 질문·정답 근거·모델을 사용하고, 근거 앞뒤의 무관한 문장 수만 늘립니다. 세 길이를 각각 세 번 호출하여 답과 입력 토큰, 정답 확인 결과를 기록합니다. 성능 저하가 나오는지는 실행 후 판단합니다. 토큰 수가 늘었다는 사실만으로 context rot가 나타났다고 결론 내리지 않습니다.

## 1. 실행 환경 준비

Python 3.12 커널에서 아래 셀을 순서대로 실행합니다. 프로젝트 폴더 또는 상위 폴더의 `.env`에 OPENAI_API_KEY를 둡니다. 이 실습은 생성 API를 총 아홉 번 호출합니다.

In [1]:
# %pip은 현재 노트북 커널에 패키지를 설치하는 Jupyter 전용 명령입니다.
# -q는 설치 로그를 줄이고, == 뒤 숫자는 수업에서 검증한 버전을 고정합니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "python-dotenv==1.2.3" "tiktoken==0.14.0"

Note: you may need to restart the kernel to use updated packages.


In [2]:
# find_dotenv는 현재 폴더부터 상위 폴더를 검색해 .env 경로를 찾습니다.
import os

from dotenv import find_dotenv, load_dotenv

# usecwd=True는 노트북을 실행한 폴더를 검색 시작점으로 지정합니다.
# 키 값은 출력하지 않고 존재 여부만 확인합니다.
env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("현재 폴더의 상위 경로에서 .env를 찾지 못했습니다.")
load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env에 OPENAI_API_KEY가 없습니다.")
print(".env와 API 키 확인 완료")

.env와 API 키 확인 완료


In [3]:
# init_chat_model은 provider:model 형식으로 채팅 모델 객체를 만듭니다.
# model은 이후 셀에서 invoke()를 호출할 때 사용합니다.
from langchain.chat_models import init_chat_model

# openai:는 공급자 이름이며, reasoning_effort="none"은 별도 추론 토큰을 사용하지 않는 설정입니다.
# max_tokens=300은 생성할 출력 토큰 수의 상한입니다.
model = init_chat_model(
    "openai:gpt-5.4-mini",
    reasoning_effort="none",
    max_tokens=300,
)
print("LangChain 모델 준비 완료")

LangChain 모델 준비 완료


## 2. 모든 조건에서 같은 문제를 사용합니다

답에 필요한 정보는 아래 근거 문장 하나에 있습니다. 모델은 휴관일과 출처를 답해야 합니다. 근거 문장과 질문을 바꾸면 문제 자체가 달라지므로 두 값을 고정합니다.

In [4]:
# evidence는 정답을 포함한 자료이며 question은 모든 실행에 사용하는 질문입니다.
evidence = "[E01 p.2] 정기 휴관일은 매주 월요일입니다."
question = "미르시립도서관의 정기 휴관일과 출처를 알려 주세요."
# note는 휴관일 답변에 필요하지 않은 주변 문장입니다.
note = "회의 준비 항목을 확인했습니다. "
print("질문:", question)
print("정답 근거:", evidence)

질문: 미르시립도서관의 정기 휴관일과 출처를 알려 주세요.
정답 근거: [E01 p.2] 정기 휴관일은 매주 월요일입니다.


## 3. 근거를 가운데에 두고 주변 문장 수만 바꿉니다

0·120·1,200은 근거의 각 앞뒤에 붙일 문장의 반복 수입니다. 토큰 수가 아닙니다. 실제 입력 토큰 수는 모델 호출 후 확인합니다. 근거를 입력 자료 가운데에 두는 배치 규칙은 세 조건에서 같습니다.

In [5]:
# 조건 이름과 근거 한쪽에 붙일 반복 수를 딕셔너리로 정합니다.
repeat_counts = {"short": 0, "medium": 120, "long": 1200}
prompts = {}
for name, count in repeat_counts.items():
    # 같은 문장을 근거 앞뒤에 같은 양으로 넣어 중앙 배치를 유지합니다.
    padding = note * count
    prompts[name] = f"""자료:
{padding}
{evidence}
{padding}

질문: {question}"""
    # 문자 수는 입력이 길어졌는지 미리 확인하는 값입니다.
    print(name, "문자 수:", len(prompts[name]))

short 문자 수: 68
medium 문자 수: 4388
long 문자 수: 43268


## 4. 비교 전에 입력 조건을 확인합니다

모든 입력에 질문과 정답 근거가 한 번씩 들어 있어야 합니다. 주변 문장에 정답을 추가하거나 근거 문장을 삭제하지 않았는지 확인합니다. 이 검사는 성능 점수가 아니라 실험에 사용할 입력의 확인입니다.

In [6]:
# 입력이 길어져도 동일한 질문과 근거가 보존되었는지 확인합니다.
for name, prompt in prompts.items():
    print(name, "근거 수:", prompt.count(evidence), "질문 수:", prompt.count(question))
    assert prompt.count(evidence) == 1
    assert prompt.count(question) == 1
# short → medium → long 순으로 실제 문자열 길이가 증가합니다.
assert len(prompts["short"]) < len(prompts["medium"]) < len(prompts["long"])

short 근거 수: 1 질문 수: 1
medium 근거 수: 1 질문 수: 1
long 근거 수: 1 질문 수: 1


## 5. 각 길이를 세 번 실행하고 실제 답을 읽습니다

짧음·중간·김 순서의 비교를 세 차례 반복합니다. `record`는 한 번 실행한 길이, 반복 번호, 답, 토큰 수를 담는 딕셔너리입니다. `records`는 아홉 개의 `record`를 차례로 모은 목록입니다. 응답의 `input_tokens`는 모델 제공자가 센 입력 토큰 수입니다. 아홉 번의 호출이 끝날 때까지 출력되는 답을 확인합니다.

In [7]:
# records는 나중에 같은 길이의 결과를 모으기 위한 기록 목록입니다.
records = []
for trial in range(1, 4):
    for name, prompt in prompts.items():
        # 모든 호출에 같은 모델·지시문을 사용합니다.
        result = model.invoke([
            ("system", "근거만 사용하고 출처를 표시하세요."),
            ("user", prompt),
        ])
        record = {
            "length": name,
            "trial": trial,
            "input_tokens": result.usage_metadata["input_tokens"],
            "answer": result.text,
        }
        records.append(record)
        print(name, "반복:", trial, "입력 토큰:", record["input_tokens"])
        print(record["answer"])

short 반복: 1 입력 토큰: 64
미르시립도서관의 **정기 휴관일은 매주 월요일**입니다.  
출처: **[E01 p.2]**


medium 반복: 1 입력 토큰: 2224
미르시립도서관의 **정기 휴관일은 매주 월요일**입니다.  
출처: **[E01 p.2]** `정기 휴관일은 매주 월요일입니다.`


long 반복: 1 입력 토큰: 21664
미르시립도서관의 정기 휴관일은 **매주 월요일**입니다.  
출처: **[E01 p.2]**


short 반복: 2 입력 토큰: 64
미르시립도서관의 **정기 휴관일은 매주 월요일**입니다.  
출처: **[E01 p.2]**


medium 반복: 2 입력 토큰: 2224
미르시립도서관의 **정기 휴관일은 매주 월요일**입니다. [E01 p.2]

**출처:** [E01 p.2]


long 반복: 2 입력 토큰: 21664
미르시립도서관의 **정기 휴관일은 매주 월요일**입니다.  
**출처:** **[E01 p.2]** `정기 휴관일은 매주 월요일입니다.`


short 반복: 3 입력 토큰: 64
미르시립도서관의 **정기 휴관일은 매주 월요일**입니다.  
출처: **[E01 p.2]**


medium 반복: 3 입력 토큰: 2224
미르시립도서관의 **정기 휴관일은 매주 월요일**입니다.  
**출처:** `[E01 p.2] 정기 휴관일은 매주 월요일입니다.`


long 반복: 3 입력 토큰: 21664
미르시립도서관의 **정기 휴관일은 매주 월요일**입니다.  
**출처:** [E01 p.2]


## 6. 모든 답에 같은 확인 기준을 적용합니다

월요일과 출처가 모두 포함되었는지 확인합니다. 이 문자열 검사만으로 답 전체의 정확성을 보장할 수는 없습니다. 위 답 원문에서 ‘월요일이 아니다’처럼 뜻이 반대인 표현, 다른 기관의 답, 근거 밖 설명이 있는지도 읽어야 합니다.

In [8]:
# 두 핵심 문자열이 모두 있으면 True, 하나라도 없으면 False입니다.
for record in records:
    answer = record["answer"]
    record["keyword_pass"] = "월요일" in answer and "E01 p.2" in answer
    print(record["length"], "반복:", record["trial"],
          "핵심어·출처 확인:", record["keyword_pass"])
# 모든 조건을 세 번씩 호출했는지 확인합니다.
assert len(records) == len(prompts) * 3

short 반복: 1 핵심어·출처 확인: True
medium 반복: 1 핵심어·출처 확인: True
long 반복: 1 핵심어·출처 확인: True
short 반복: 2 핵심어·출처 확인: True
medium 반복: 2 핵심어·출처 확인: True
long 반복: 2 핵심어·출처 확인: True
short 반복: 3 핵심어·출처 확인: True
medium 반복: 3 핵심어·출처 확인: True
long 반복: 3 핵심어·출처 확인: True


## 7. 입력 토큰과 정답 확인 결과를 나란히 비교합니다

각 길이에서 조건을 충족한 응답 수를 셉니다. 점수가 낮아졌다면 먼저 실제 답을 다시 읽습니다. 점수가 같다면 이번 과제와 입력 범위에서는 이 기준으로 성능 저하를 관찰하지 못한 것입니다.

`rows`는 현재 길이 조건과 이름이 같은 기록만 고르는 목록 표현식입니다. `sum()`은 참을 1, 거짓을 0으로 더하여 문자열 기준 충족 횟수를 셉니다. 이번에는 길이마다 기록이 세 개이므로 분모가 3입니다. 이 값은 답 전체의 사실성을 자동 채점한 점수가 아닙니다.

회의록 담당자 찾기에도 같은 비교 방법을 적용할 수 있습니다. 여러 문서의 일정을 연결하는 과제는 필요한 추론이 달라 별도 실험이 필요합니다. 추가 실험에서 근거 위치와 길이를 동시에 바꾸면 원인을 구분하기 어려우므로 하나씩 변경합니다. 여기의 저장 출력에는 그런 추가 실험이 포함되어 있지 않습니다.

In [9]:
# rows는 같은 입력 길이 조건의 실행 기록만 모은 목록입니다.
for name in prompts:
    rows = [record for record in records if record["length"] == name]
    passed = sum(record["keyword_pass"] for record in rows)
    # 토큰 수는 세 호출의 값을 모두 보여 주어 실행 기록을 보존합니다.
    tokens = [record["input_tokens"] for record in rows]
    print(name, "입력 토큰:", tokens, "기준 충족:", passed, "/", len(rows))

short 입력 토큰: [64, 64, 64] 기준 충족: 3 / 3
medium 입력 토큰: [2224, 2224, 2224] 기준 충족: 3 / 3
long 입력 토큰: [21664, 21664, 21664] 기준 충족: 3 / 3


## 결과 해석

입력 토큰 증가와 정답 확인 결과를 함께 읽습니다. 모든 길이에서 정답을 유지할 수도 있습니다. 그런 결과를 성능 저하가 발생한 것처럼 표현하지 않습니다. 짧은 근거 한 문장과 반복된 주변 문장만 사용하는 작은 실험이므로, 긴 실제 문서의 모든 과제로 결론을 확대하지 않습니다. 더 복잡한 과제와 여러 질문을 사용하는 평가는 Ch 11·15에서 다룹니다.